In [20]:
# ============================================================
# FINAL OPTIMIZED LVEDP PREDICTION MODEL ON 13 FEATURES
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import NearestNeighbors

# =======================
# 1) LOAD DATA
# =======================
PATH = r"D:\Nti\graduation_project\1-10-2005.xlsx"
df = pd.read_excel(PATH)

# =======================
# 2) DEFINE TARGET AND FEATURES
# =======================
target = [c for c in df.columns if "lvedp" in c.lower()][0]

features = [
    "Mitral E velocity (cm/s)",
    "Septal é velocity (cm/s)",
    "Lateral é velocity (cm/s)",
    "E/E`",
    "Indexed LA volume (mL/m²)",
    "Estimated mean LA pressure ",
    "TR velocity (m/s)",
    "PASP (mmHg)",
    "LVEF (%)",
    "LV global long. Strain (%)",
    "LA reservoir (%)",
    "LA conduit (%)",
    "LA contraction (%)"
]

# Drop rows with missing target
df = df.dropna(subset=[target])

# Ensure all features are numeric
for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].astype(float)
    df[col].fillna(df[col].median(), inplace=True)

X = df[features].reset_index(drop=True)
y = df[target].reset_index(drop=True)

# =======================
# 3) APPLY ENHANCED SMOTE
# =======================
target_bins = 4
y_bins = pd.qcut(y, q=target_bins, labels=False, duplicates='drop')
bin_counts = pd.Series(y_bins).value_counts()
max_samples = max(bin_counts.max(), 50)

sampling_strategy = {bin_idx: max_samples for bin_idx in bin_counts.index}
safe_k = min(3, bin_counts.min() - 1)
if safe_k < 1: safe_k = 1

sm = SMOTE(k_neighbors=safe_k, sampling_strategy=sampling_strategy, random_state=42)
X_sm, y_bins_sm = sm.fit_resample(X, y_bins)

# Reconstruct continuous target for synthetic samples
nn = NearestNeighbors(n_neighbors=3).fit(X)
y_sm = y.values.copy()
for idx in range(len(X), len(X_sm)):
    synthetic_sample = X_sm[idx:idx+1]
    distances, neighbor_idx = nn.kneighbors(synthetic_sample)
    neighbor_targets = y.values[neighbor_idx[0]]
    weights = 1 / (distances[0] + 1e-8)
    weights /= weights.sum()
    synthetic_target = np.dot(weights, neighbor_targets)
    y_sm = np.append(y_sm, synthetic_target)

# =======================
# 4) SCALE FEATURES
# =======================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sm)

# =======================
# 5) TRAIN-TEST SPLIT
# =======================
y_bins_cont = pd.qcut(y_sm, q=4, labels=False, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_sm, test_size=0.33, random_state=42, stratify=y_bins_cont
)

# =======================
# 6) CREATE OPTIMIZED ENSEMBLE
# =======================
def create_ensemble():
    base_models = [
        ('rf', RandomForestRegressor(n_estimators=100, max_depth=5,
                                     min_samples_split=10, min_samples_leaf=5,
                                     max_features=0.6, random_state=42)),
        ('gbm', GradientBoostingRegressor(n_estimators=150, max_depth=3,
                                          learning_rate=0.05, subsample=0.8,
                                          max_features=0.7, random_state=42)),
        ('ridge', Ridge(alpha=5.0, random_state=42)),
        ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale'))
    ]
    ensemble = VotingRegressor(base_models, weights=[1,1,1,1], n_jobs=1)
    return ensemble

ensemble = create_ensemble()
ensemble.fit(X_train, y_train)

# =======================
# 7) EVALUATION
# =======================
train_pred = ensemble.predict(X_train)
test_pred = ensemble.predict(X_test)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

print(f"Train MAE: {train_mae:.4f}, Test MAE: {test_mae:.4f}")
print(f"Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")

# =======================
# 8) STRATIFIED CROSS-VALIDATION
# =======================
def stratified_cv(X, y, model, n_splits=5):
    y_bins = pd.qcut(y, q=4, labels=False, duplicates='drop')
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for train_idx, test_idx in skf.split(X, y_bins):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        model.fit(X_tr, y_tr)
        scores.append(mean_absolute_error(y_te, model.predict(X_te)))
    return np.mean(scores), np.std(scores)

cv_mean, cv_std = stratified_cv(X_scaled, y_sm, ensemble)
print(f"Cross-validation MAE: {cv_mean:.4f} ± {cv_std:.4f}")

# =======================
# 9) SAVE MODEL
# =======================
model_data = {
    "model": ensemble,
    "scaler": scaler,
    "features": features,
    "target": target
}
joblib.dump(model_data, "optimized_lvedp_model_13_features.joblib")
print("💾 Model saved: optimized_lvedp_model_13_features.joblib")


Train MAE: 0.9926, Test MAE: 1.0655
Train R²: 0.9046, Test R²: 0.8846
Cross-validation MAE: 1.3264 ± 0.0586
💾 Model saved: optimized_lvedp_model_13_features.joblib
